In [23]:
import zipfile
import xarray as xr
import io
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import numpy as np
import tempfile
import os
import pandas as pd

In [2]:
var_list = [
    "mean_surface_direct_short_wave_radiation_flux_A",
    "mean_surface_direct_short_wave_radiation_flux_B",
    "mean_surface_direct_short_wave_radiation_flux_C",
    "mean_surface_direct_short_wave_radiation_flux_D",
    "mean_surface_direct_short_wave_radiation_flux_E",
    "mean_surface_direct_short_wave_radiation_flux_F"
]

In [ ]:
file = "./data_output/mean_surface_direct_short_wave_radiation_flux_A_2006-2016/data.grib"

dado = xr.open_dataset(file, engine="cfgrib", decode_times=True, decode_timedelta=True)
# criar valid_time como coordenada
# 1. Criar valid_time como DataArray (broadcast sobre step)
valid_time = dado["time"] + dado["step"]
dado = dado.assign_coords(valid_time=(("time", "step"), valid_time.data))

dado_step_mean = dado.mean(dim="step")
dado_diario = dado_step_mean.resample(time="1D").mean()
# dado = dado.resample(time="1D").mean()

surface_downwelling_shortwave = dado_diario.to_dataframe().reset_index()
surface_downwelling_shortwave.head()

In [4]:
for var in var_list:
    zip_files_years = [i for i in os.listdir("./data_output/") if var in i and '.zip' in i]
zip_files_years

['mean_surface_direct_short_wave_radiation_flux_F_2017-2020.zip',
 'mean_surface_direct_short_wave_radiation_flux_F_2006-2016.zip']

In [6]:
zip_files_years[0]

'mean_surface_direct_short_wave_radiation_flux_F_2017-2020.zip'

In [19]:
def df_generator_data_grib(zip_file):
    inner_filename = "data.grib"

    with zipfile.ZipFile(zip_file, "r") as zf:

        with zf.open(inner_filename) as f:
            with tempfile.NamedTemporaryFile(suffix=".grib", delete=False) as tmp:
                    tmp.write(f.read())
                    tmp_path = tmp.name

    dado = xr.open_dataset(tmp_path, engine="cfgrib", decode_times=True, decode_timedelta=True)

    valid_time = dado["time"] + dado["step"]
    dado = dado.assign_coords(valid_time=(("time", "step"), valid_time.data))

    dado_step_mean = dado.mean(dim="step")
    dado_diario = dado_step_mean.resample(time="1D").mean()

    dado_diario_df = dado_diario.to_dataframe().reset_index()
    return(dado_diario_df)


In [24]:
def concat(df1,df2):
    df_final = pd.concat([df1, df2], ignore_index=True)
    return(df_final)

In [ ]:
teste = df_generator_data_grib("./data_output/"+zip_files_years[0])
teste.head()


,time,latitude,longitude,avg_sdirswrf,number,surface
0,2016-12-31,-2.501,-35.0,0.000000,0,0.0
1,2017-01-01,-2.501,-35.0,156.180405,0,0.0
2,2017-01-02,-2.501,-35.0,4.563933,0,0.0
3,2017-01-03,-2.501,-35.0,48.290535,0,0.0
4,2017-01-04,-2.501,-35.0,193.769470,0,0.0


In [22]:
teste['point'] = zip_files_years[0].split('_')[-2]
teste.head()

,time,latitude,longitude,avg_sdirswrf,number,surface,point
0,2016-12-31,-2.501,-35.0,0.000000,0,0.0,F
1,2017-01-01,-2.501,-35.0,156.180405,0,0.0,F
2,2017-01-02,-2.501,-35.0,4.563933,0,0.0,F
3,2017-01-03,-2.501,-35.0,48.290535,0,0.0,F
4,2017-01-04,-2.501,-35.0,193.769470,0,0.0,F


In [25]:
teste2 = concat(teste,teste)
len(teste2)

2924

In [ ]:
# # Exportando para CSV
# df.to_csv("dados_saida.csv", index=False, sep=";")

In [ ]:
inner_filename = "data.grib"

# Abrindo o ZIP em modo leitura
with zipfile.ZipFile("./data_output/"+zip_files_years[0], "r") as zf:
    # Lendo o conteúdo do arquivo interno em bytes
    with zf.open(inner_filename) as f:
       with tempfile.NamedTemporaryFile(suffix=".grib", delete=False) as tmp:
            tmp.write(f.read())
            tmp_path = tmp.name  # caminho físico para o arquivo temporário

# Agora o cfgrib pode abrir corretamente
dado = xr.open_dataset(tmp_path, engine="cfgrib", decode_times=True, decode_timedelta=True)

valid_time = dado["time"] + dado["step"]
dado = dado.assign_coords(valid_time=(("time", "step"), valid_time.data))

dado_step_mean = dado.mean(dim="step")
dado_diario = dado_step_mean.resample(time="1D").mean()
# dado = dado.resample(time="1D").mean()

dado_diario_df = dado_diario.to_dataframe().reset_index()
dado_diario_df['point'] = zip_files_years[0].split('_')[-2]
dado_diario_df.head()

,time,latitude,longitude,avg_sdirswrf,number,surface
0,2016-12-31,-2.501,-35.0,0.000000,0,0.0
1,2017-01-01,-2.501,-35.0,156.180405,0,0.0
2,2017-01-02,-2.501,-35.0,4.563933,0,0.0
3,2017-01-03,-2.501,-35.0,48.290535,0,0.0
4,2017-01-04,-2.501,-35.0,193.769470,0,0.0


In [13]:
len(dado_diario_df)

1462

In [ ]:
dado_diario_df['point'] = 'F'